# Resampling of Flair

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

### Read Existing Tables

In [2]:
file_path = "../../data/out/FLAIRPublicDataSet/"
file_map = {
    'cgm': 'Flair_cgm_history.csv.gz',
    'bolus': 'Flair_bolus_event_history.csv.gz',
    'basal': 'Flair_basal_event_history.csv.gz',
}

In [3]:
def get_df_from_file(file_path, file_name, parse_datetime=True, sep=','):
    df = pd.read_csv(file_path + file_name, sep=sep)
    if parse_datetime:
        df['date'] = pd.to_datetime(df['datetime'], unit='s')
    return df

In [4]:
def get_extended_df(original_df, value_column):
    """
    Get a df where quantities are distributed throughout 5-minute intervals instead of having start- and end dates.
    """
    new_rows = []
    for _, row in original_df.iterrows():
        new_rows.extend(split_duration(row, value_column))
    extended_df = pd.DataFrame(new_rows)
    extended_df.set_index('date', inplace=True)
    return extended_df

def split_duration(row, value_column):
    """
    For features with a duration, we split the values across 5-minute intervals by adding
    new rows for every 5-minute window in duration, and equally split the original quantity across those rows.
    """
    duration = row['end_date'] - row['date']
    rounded_duration = round(duration / pd.Timedelta(minutes=5)) * pd.Timedelta(minutes=5)
    num_intervals = rounded_duration // pd.Timedelta(minutes=5)
    if num_intervals < 1:
        num_intervals = 1
    value_per_interval = row[value_column] / num_intervals
    new_rows = []
    for i in range(int(num_intervals)):
        new_row = {
            'date': row['date'] + pd.Timedelta(minutes=5 * i),
            value_column: value_per_interval,
            'patient_id': row['patient_id'],
        }
        new_rows.append(new_row)
    return new_rows

In [5]:
def parse_flair_dates(dates, format_date = '%m/%d/%Y', format_time = '%I:%M:%S %p'):
    """Parse date strings separately for those with/without time component, interpret those without as midnight (00AM)
    Args:
        dates (pd.DataFrame): datetimes (string) either in in the %m/%d/%Y or %m/%d/%Y %I:%M:%S %p format
    Returns:
        pandas series: with parsed dates
    """
    #make sure to only parse dates if the value is not null
    dates = dates.astype(str)
    only_date = dates.apply(len) <=10
    dates_copy = dates.copy()
    dates_copy.loc[only_date] = pd.to_datetime(dates.loc[only_date], format=format_date)
    dates_copy.loc[~only_date] = pd.to_datetime(dates.loc[~only_date], format=f'{format_date} {format_time}')
    return dates_copy.astype('datetime64[ns]')

In [6]:
df_glucose = get_df_from_file(file_path, file_map['cgm'])
df_bolus = get_df_from_file(file_path, file_map['bolus'])
df_basal = get_df_from_file(file_path, file_map['basal'])

### Resample Existing Tables

In [7]:
df_glucose.set_index('date', inplace=True)
df_glucose.head()

,patient_id,datetime,cgm
date,,,
2019-05-15 21:18:15,26,1557955095,136
2019-05-15 21:13:15,26,1557954795,146
2019-05-15 21:08:15,26,1557954495,157
2019-05-15 21:03:15,26,1557954195,166
2019-05-15 20:58:15,26,1557953895,174


In [8]:
df_bolus_orig = df_bolus.copy()
df_bolus['end_date'] = df_bolus['date'] + pd.to_timedelta(df_bolus['delivery_duration'], unit='s')
df_bolus = get_extended_df(df_bolus, 'bolus')
df_bolus

,bolus,patient_id
date,,
2018-06-04 13:33:20,7.7,88
2018-06-04 14:03:02,2.2,88
2018-06-04 19:08:38,6.6,88
2018-06-04 20:42:51,0.3,88
2018-06-04 23:01:28,5.1,88
...,...,...
2020-03-29 14:12:48,5.3,4
2020-03-29 18:37:00,10.4,4
2020-03-30 15:20:20,4.6,4


In [9]:
print(f'New sum after distribution of extended boluses: {df_bolus["bolus"].sum():.2f}, should be: {df_bolus_orig["bolus"].sum():.2f}')

New sum after distribution of extended boluses: 1235667.82, should be: 1235667.82


In [10]:
df_basal_orig = df_basal.copy()
df_basal.sort_values(by=['patient_id', 'date'], inplace=True)
df_basal.set_index('date', inplace=True)
df_basal

,patient_id,datetime,basal_rate
date,,,
2018-08-08 14:10:00,1,1533737400,0.00
2018-08-08 15:11:00,1,1533741060,1.30
2018-08-08 15:18:10,1,1533741490,0.00
2018-08-08 15:22:22,1,1533741742,1.30
2018-08-08 16:08:38,1,1533744518,0.00
...,...,...,...
2019-05-27 18:57:31,126,1558983451,0.00
2019-05-27 19:23:26,126,1558985006,1.05
2019-05-27 22:00:00,126,1558994400,1.25


In [11]:
processed_dfs = []
subject_ids = df_glucose['patient_id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_glucose[df_glucose['patient_id'] == subject_id].copy()
    df_subject = df_subject[['cgm']].resample('5min', label='right').mean()
    df_subject['patient_id'] = subject_id
    df_subject.sort_index(inplace=True)

    def merge_data(df_col, df_subject, col_names, subject_id, agg_type='sum'):
        """ agg_type is data aggregation type. """
        df_subset = df_col[df_col['patient_id'] == subject_id].copy()
        if not df_subset.empty:
            if agg_type == 'mean':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').mean()
            elif agg_type == 'sum':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            elif agg_type == 'first':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').first()
            elif agg_type == 'last':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').last()
            else:
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            df_subject = pd.merge(df_subject, df_subset, on="date", how='outer')
        else:
            df_subject[col_names] = np.nan
        return df_subject

    # Add insulin and insulin type
    df_subject = merge_data(df_bolus, df_subject, ['bolus'], subject_id, agg_type='sum')
    df_subject = merge_data(df_basal, df_subject, ['basal_rate'], subject_id, agg_type='last')
    df_subject['basal_rate'] = df_subject['basal_rate'].ffill()
    
    df_subject['patient_id'] = subject_id
    df_subject = df_subject.rename(columns={'patient_id': 'id', 'basal_rate': 'basal', 'cgm': 'CGM'})
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)

Subjects: 113
26 is finished processing
40 is finished processing
78 is finished processing
46 is finished processing
15 is finished processing
33 is finished processing
37 is finished processing
85 is finished processing
36 is finished processing
20 is finished processing
126 is finished processing
11 is finished processing
97 is finished processing
73 is finished processing
120 is finished processing
79 is finished processing
16 is finished processing
14 is finished processing
81 is finished processing
88 is finished processing
105 is finished processing
13 is finished processing
95 is finished processing
99 is finished processing
48 is finished processing
18 is finished processing
56 is finished processing
90 is finished processing
93 is finished processing
9 is finished processing
72 is finished processing
102 is finished processing
115 is finished processing
69 is finished processing
23 is finished processing
86 is finished processing
84 is finished processing
38 is finished proce

In [12]:
df_final

,CGM,id,bolus,basal
date,,,,
2019-02-08 14:05:00,NaN,26,NaN,1.15
2019-02-08 14:10:00,NaN,26,NaN,1.15
2019-02-08 14:15:00,NaN,26,NaN,1.15
2019-02-08 14:20:00,NaN,26,NaN,1.15
2019-02-08 14:25:00,NaN,26,NaN,1.15
...,...,...,...,...
2019-07-26 08:00:00,171.0,113,0.15,0.00
2019-07-26 08:05:00,180.0,113,0.45,0.00
2019-07-26 08:10:00,185.0,113,0.13,0.00


### Add Additional Tables

We found:
- Insulin type
- Age
- Weight
- Height
- Gender

In [13]:
raw_data_file_path = "../../data/raw/FLAIRPublicDataSet/Data Tables/"

In [14]:
df_insulin_type = get_df_from_file(raw_data_file_path, 'FLAIRInsulin.txt', parse_datetime=False, sep='|')
df_insulin_type = df_insulin_type[['PtID', 'InsulinName']]
df_insulin_type = df_insulin_type.rename(columns={'PtID': 'id', 'InsulinName': 'insulin_type'})
df_insulin_type

,id,insulin_type
0,103,Humalog (Lispro)
1,88,Novolog (Aspart)
2,105,Novolog (Aspart)
3,13,Novolog (Aspart)
4,31,Novolog (Aspart)
...,...,...
191,14,Humalog (Lispro)
192,48,Humalog (Lispro)
193,86,Novolog (Aspart)
194,30,Novolog (Aspart)


In [15]:
def add_single_value_to_subjects(df, df_new_val, col_name):
    # TODO: this function is very inefficient... add value directly to located rows instead
    processed_dfs = []
    subject_ids = df['id'].unique()
    for subject_id in subject_ids:
        df_subject = df[df['id'] == subject_id].copy()
        df_subject.sort_index(inplace=True)
    
        user_data = df_new_val[df_new_val['id'] == subject_id].copy()
        if not user_data.empty:
            if pd.api.types.is_numeric_dtype(user_data[col_name]):
                print(f"Mean val from {len(user_data[col_name])} values: ", user_data[col_name].mean(skipna=True))
                df_subject[col_name] = user_data[col_name].mean(skipna=True)
            else:
                print(f"String val with {len(user_data[col_name])}: ", ", ".join(user_data[col_name].dropna().astype(str)))
                df_subject[col_name] = ", ".join(user_data[col_name].dropna().astype(str))  # Take the first value for non-numeric data
        else:
            df_subject[col_name] = np.nan        
        processed_dfs.append(df_subject)
        
    df = pd.concat(processed_dfs)
    return df

In [16]:
# TODO in future work: should we handle MDI? How? 
# TODO: When subjects change insulin we should also handle that! 
# TODO: I think that insulin type should be a part of the bolus / basal tables, so that we can specify at that point

In [17]:
# Add insulin type
df_final = add_single_value_to_subjects(df_final, df_insulin_type, 'insulin_type')
df_final

String val with 1:  Novolog (Aspart)
String val with 2:  Novolog Fiasp, Humalog (Lispro)
String val with 1:  Novolog (Aspart)
String val with 3:  Humalog (Lispro), Humalog (Lispro), Humalog (Lispro)
String val with 1:  Humalog (Lispro)
String val with 2:  Humalog (Lispro), Humalog (Lispro)
String val with 1:  Humalog (Lispro)
String val with 1:  Humalog (Lispro)
String val with 1:  Novolog (Aspart)
String val with 1:  Humalog (Lispro)
String val with 3:  Novolog (Aspart), Humalog (Lispro), Humalog (Lispro)
String val with 1:  Humalog (Lispro)
String val with 1:  Humalog (Lispro)
String val with 1:  Humalog (Lispro)
String val with 2:  Novolog (Aspart), Humalog (Lispro)
String val with 3:  Degludec (Tresiba), Humalog (Lispro), Humalog (Lispro)
String val with 1:  Humalog (Lispro)
String val with 6:  Humalog (Lispro), Degludec (Tresiba), Humalog (Lispro), Humalog (Lispro), Degludec (Tresiba), Humalog (Lispro)
String val with 4:  Humalog (Lispro), Lantus (Glargine) 1 time per day, Humalog

,CGM,id,bolus,basal,insulin_type
date,,,,,
2019-02-08 14:05:00,NaN,26,NaN,1.15,Novolog (Aspart)
2019-02-08 14:10:00,NaN,26,NaN,1.15,Novolog (Aspart)
2019-02-08 14:15:00,NaN,26,NaN,1.15,Novolog (Aspart)
2019-02-08 14:20:00,NaN,26,NaN,1.15,Novolog (Aspart)
2019-02-08 14:25:00,NaN,26,NaN,1.15,Novolog (Aspart)
...,...,...,...,...,...
2019-07-26 08:00:00,171.0,113,0.15,0.00,Humalog (Lispro)
2019-07-26 08:05:00,180.0,113,0.45,0.00,Humalog (Lispro)
2019-07-26 08:10:00,185.0,113,0.13,0.00,Humalog (Lispro)


In [18]:
# Add age
df_age = get_df_from_file(raw_data_file_path, 'PtRoster.txt', parse_datetime=False, sep='|')
df_age = df_age[['PtID', 'AgeAsofEnrollDt']]
df_age = df_age.rename(columns={'PtID': 'id', 'AgeAsofEnrollDt': 'age'})
df_age

,id,age
0,32,14
1,26,18
2,40,20
3,78,20
4,46,14
...,...,...
121,104,15
122,67,20
123,43,21
124,68,20


In [19]:
# Add age
df_final = add_single_value_to_subjects(df_final, df_age, 'age')
df_final.head()

Mean val from 1 values:  18.0
Mean val from 1 values:  20.0
Mean val from 1 values:  20.0
Mean val from 1 values:  14.0
Mean val from 1 values:  15.0
Mean val from 1 values:  20.0
Mean val from 1 values:  16.0
Mean val from 1 values:  25.0
Mean val from 1 values:  16.0
Mean val from 1 values:  21.0
Mean val from 1 values:  14.0
Mean val from 1 values:  23.0
Mean val from 1 values:  20.0
Mean val from 1 values:  25.0
Mean val from 1 values:  26.0
Mean val from 1 values:  23.0
Mean val from 1 values:  24.0
Mean val from 1 values:  17.0
Mean val from 1 values:  14.0
Mean val from 1 values:  15.0
Mean val from 1 values:  23.0
Mean val from 1 values:  16.0
Mean val from 1 values:  14.0
Mean val from 1 values:  16.0
Mean val from 1 values:  29.0
Mean val from 1 values:  26.0
Mean val from 1 values:  16.0
Mean val from 1 values:  24.0
Mean val from 1 values:  27.0
Mean val from 1 values:  17.0
Mean val from 1 values:  15.0
Mean val from 1 values:  15.0
Mean val from 1 values:  26.0
Mean val f

,CGM,id,bolus,basal,insulin_type,age
date,,,,,,
2019-02-08 14:05:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0
2019-02-08 14:10:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0
2019-02-08 14:15:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0
2019-02-08 14:20:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0
2019-02-08 14:25:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0


In [20]:
# Add gender
df_gender = get_df_from_file(raw_data_file_path, 'FLAIRDiabScreening.txt', parse_datetime=False, sep='|')
df_gender = df_gender[['PtID', 'Sex']]
df_gender = df_gender.rename(columns={'PtID': 'id', 'Sex': 'gender'})
df_gender

,id,gender
0,103,F
1,105,F
2,88,M
3,31,F
4,83,F
...,...,...
114,62,M
115,113,F
116,59,F
117,22,M


In [21]:
# Add weight, height, and gender
df_final = add_single_value_to_subjects(df_final, df_gender, 'gender')
df_final.head()

String val with 1:  F
String val with 1:  F
String val with 1:  M
String val with 1:  F
String val with 1:  M
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  M
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  F
String val with 1:  M
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  M
String val with 1:  F
String val with 1:  F
String val with 1:  M
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  M
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val with 1:  F
String val

,CGM,id,bolus,basal,insulin_type,age,gender
date,,,,,,,
2019-02-08 14:05:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F
2019-02-08 14:10:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F
2019-02-08 14:15:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F
2019-02-08 14:20:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F
2019-02-08 14:25:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F


In [22]:
# Add weight and height
df_weight_height = get_df_from_file(raw_data_file_path, 'FLAIRDiabPhysExam.txt', parse_datetime=False, sep='|')

print(df_weight_height['HeightUnits'].unique())
print(df_weight_height['WeightUnits'].unique())

# If inches or lbs, --> cm and kg
df_weight_height['Height'] = df_weight_height.apply(
    lambda row: row['Height'] * 2.54 if row['HeightUnits'] != 'cm' else row['Height'],
    axis=1
)
df_weight_height['Weight'] = df_weight_height.apply(
    lambda row: row['Weight'] * 0.453592 if row['WeightUnits'] != 'kg' else row['Weight'],
    axis=1
)
df_weight_height[['Weight', 'Height']] = df_weight_height[['Weight', 'Height']].round(1)
df_weight_height = df_weight_height[['PtID', 'Weight', 'Height', 'Visit']]
df_weight_height = df_weight_height.rename(columns={'PtID': 'id', 'Weight': 'weight', 'Height': 'height'})
df_weight_height

['cm' 'in' nan]
['lbs' 'kg' nan]


,id,weight,height,Visit
0,103,82.1,167.0,Screening
1,105,75.3,163.1,Screening
2,88,57.6,170.0,Screening
3,31,82.1,159.0,Screening
4,83,120.4,182.5,Screening
...,...,...,...,...
450,38,61.4,164.0,End of Study
451,4,80.1,174.0,End of Study
452,7,112.2,181.1,End of Study
453,39,67.9,166.0,End of Study


In [23]:
df_weight_height[df_weight_height.duplicated('id', keep=False)].sort_values(by=['id', 'Visit'])

,id,weight,height,Visit
414,1,64.8,168.0,End of Study
177,1,60.2,168.0,Period 1: Closed Loop Treatment Initiation
302,1,59.0,168.0,Period 2: Closed Loop Treatment Initiation
146,1,60.5,168.0,Screening
393,2,69.1,177.8,End of Study
...,...,...,...,...
30,125,58.0,155.0,Screening
420,126,68.1,174.4,End of Study
188,126,60.4,173.6,Period 1: Closed Loop Treatment Initiation
313,126,65.7,174.1,Period 2: Closed Loop Treatment Initiation


In [24]:
# TODO in future work: For each subject, the weight and height and age should be given for the correct dates! and updated when measurements are updated!

In [25]:
# Add weight, height, and gender
df_final = add_single_value_to_subjects(df_final, df_weight_height, 'weight')
df_final = add_single_value_to_subjects(df_final, df_weight_height, 'height')
df_final[['weight', 'height']] = df_final[['weight', 'height']].round(1)
df_final.head()

Mean val from 4 values:  82.775
Mean val from 4 values:  76.80000000000001
Mean val from 4 values:  75.625
Mean val from 4 values:  75.275
Mean val from 4 values:  51.625
Mean val from 3 values:  99.8
Mean val from 4 values:  57.224999999999994
Mean val from 4 values:  67.55
Mean val from 4 values:  90.125
Mean val from 4 values:  76.825
Mean val from 4 values:  63.199999999999996
Mean val from 4 values:  85.975
Mean val from 4 values:  66.0
Mean val from 4 values:  83.7
Mean val from 4 values:  74.375
Mean val from 4 values:  82.225
Mean val from 4 values:  127.07500000000002
Mean val from 4 values:  92.575
Mean val from 4 values:  60.275000000000006
Mean val from 4 values:  59.65
Mean val from 4 values:  77.07499999999999
Mean val from 4 values:  81.675
Mean val from 4 values:  59.0
Mean val from 4 values:  85.30000000000001
Mean val from 4 values:  83.94999999999999
Mean val from 4 values:  60.825
Mean val from 4 values:  64.75
Mean val from 4 values:  72.65
Mean val from 4 values: 

,CGM,id,bolus,basal,insulin_type,age,gender,weight,height
date,,,,,,,,,
2019-02-08 14:05:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:10:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:15:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:20:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:25:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8


### Save Resampled Data

In [26]:
df_final

,CGM,id,bolus,basal,insulin_type,age,gender,weight,height
date,,,,,,,,,
2019-02-08 14:05:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:10:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:15:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:20:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
2019-02-08 14:25:00,NaN,26,NaN,1.15,Novolog (Aspart),18.0,F,82.8,163.8
...,...,...,...,...,...,...,...,...,...
2019-07-26 08:00:00,171.0,113,0.15,0.00,Humalog (Lispro),17.0,F,72.8,172.0
2019-07-26 08:05:00,180.0,113,0.45,0.00,Humalog (Lispro),17.0,F,72.8,172.0
2019-07-26 08:10:00,185.0,113,0.13,0.00,Humalog (Lispro),17.0,F,72.8,172.0


In [27]:
save_file_path = "../../data/resampled/"
os.makedirs(save_file_path, exist_ok=True)
df_final.to_csv(save_file_path + 'Flair.csv')